In [ ]:
import sys
from pathlib import Path
import torch

script_dir = Path.cwd()
if not (script_dir / "utils" / "sls_controlled_experiment.py").exists():
    candidate = Path("ad_detection/train_notebook/SLS").resolve()
    if candidate.exists():
        script_dir = candidate
sys.path.insert(0, str(script_dir.parent.parent / "train"))
sys.path.insert(0, str(script_dir / "utils"))

from utils.config import PROJECT_ROOT
from utils.data_split import create_split
from SLS_Model.extract_feature import extract_feature
from utils.visualization import plot_training_curves
from utils.dataset import create_dataloaders
from SLS_Model.train import train, validate
from sls_controlled_experiment import (
    bootstrap_binary_metrics_with_samples,
    compute_binary_metrics,
    evaluate_lu_with_predictions,
    flatten_bootstrap_summary,
    load_dict_rows,
    save_dict_rows,
    summarize_metric_rows,
)
import numpy as np

SLS_BATCH_SIZE = 4
BOOTSTRAP_REPEATS = 1000
LU_SAMPLE_SIZE = 74


In [ ]:
DATASET_NAME = "ADReSS"
data_version = "raw"

In [ ]:
# RAW_AUDIO_DIR = PROJECT_ROOT / f"data/denoised/{DATASET_NAME}"
RAW_AUDIO_DIR = PROJECT_ROOT / f"data/{data_version}/{DATASET_NAME}"

SLS_FEATURES_DIR = PROJECT_ROOT / f"data/processed/{DATASET_NAME}_sls_features"
FEATURE_DIR_NAME = f"{DATASET_NAME}_sls_features"
MODEL_OUTPUT_DIR = PROJECT_ROOT / f"models/{DATASET_NAME}_sls_multi_seed"
LU_RESULTS_DIR = MODEL_OUTPUT_DIR / "lu_test_results"
TRAINING_RESULTS_PATH = LU_RESULTS_DIR / "training_results.csv"

In [ ]:
if torch.cuda.is_available():
      device = torch.device('cuda')
      accelerator = 'gpu'
elif torch.backends.mps.is_available():
      device = torch.device('mps')
      accelerator = 'mps'
else:
      device = torch.device('cpu')
      accelerator = 'cpu'
print(f"Using {device}")

## Step 1: Train/Validation Set Split

In [ ]:
TRAIN_CSV, VAL_CSV = create_split(DATASET_NAME, feature_type='sls')

## Step 2: Extract SLS Features (all 24 XLS-R layers, fp16)

In [ ]:
ssl_model = extract_feature(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    raw_audio_dir=RAW_AUDIO_DIR,
    sls_features_dir=SLS_FEATURES_DIR,
    device=device,
)

## Step 4: Create Data Loaders

In [ ]:
train_loader = create_dataloaders(data_csv=TRAIN_CSV, feature_type='sls', batch_size=SLS_BATCH_SIZE)

val_loader = create_dataloaders(data_csv=VAL_CSV, feature_type='sls', batch_size=SLS_BATCH_SIZE)

## Step 5: Define Training Function and Model

In [ ]:
all_results = {
    'seeds': [],
    'val_accs': [],
    'val_losses': [],
    'control_accs': [],
    'dementia_accs': [],
    'macro_f1_scores': [],
    'control_f1_scores': [],
    'dementia_f1_scores': [],
}


def record_seed_result(seed, metrics):
    if seed in all_results['seeds']:
        return
    all_results['seeds'].append(seed)
    all_results['val_accs'].append(metrics['val_acc'])
    all_results['val_losses'].append(metrics['val_loss'])
    all_results['control_accs'].append(metrics['control_acc'])
    all_results['dementia_accs'].append(metrics['dementia_acc'])
    all_results['macro_f1_scores'].append(metrics['macro_f1'])
    all_results['control_f1_scores'].append(metrics['control_f1'])
    all_results['dementia_f1_scores'].append(metrics['dementia_f1'])


def evaluate_seed_metrics(seed, val_loader, device):
    import torch.nn.functional as F
    from SLS_Model.model import AD_SLS_Model

    best_model_path = MODEL_OUTPUT_DIR / f"seed_{seed}" / "best.pth"
    model = AD_SLS_Model().to(device)
    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint)
    model.eval()

    total_loss = 0
    y_true = []
    y_pred = []
    with torch.no_grad():
        for features, labels, masks in val_loader:
            features = features.to(device)
            labels = labels.to(device)
            masks = masks.to(device)
            logits = model(features, masks)
            total_loss += F.cross_entropy(logits, labels).item()
            predictions = torch.argmax(logits, dim=1)
            y_true.extend(labels.detach().cpu().tolist())
            y_pred.extend(predictions.detach().cpu().tolist())

    metrics = compute_binary_metrics(y_true, y_pred)
    del model
    return {
        'val_acc': metrics['accuracy'],
        'val_loss': total_loss / len(val_loader),
        'control_acc': metrics['control_acc'],
        'dementia_acc': metrics['dementia_acc'],
        'macro_f1': (metrics['control_f1'] + metrics['dementia_f1']) / 2,
        'control_f1': metrics['control_f1'],
        'dementia_f1': metrics['dementia_f1'],
        'f1_score': metrics['dementia_f1'],
    }


def train_or_recover_seed(seed, train_loader, val_loader, output_dir, device):
    best_model_path = output_dir / f"seed_{seed}" / "best.pth"
    history = None
    if best_model_path.exists():
        print(f"Skip training seed {seed}: found existing best model at {best_model_path}")
    else:
        _, _, history = train(
            seed=seed,
            train_loader=train_loader,
            val_loader=val_loader,
            output_dir=output_dir,
            device=device,
        )

    metrics = evaluate_seed_metrics(seed, val_loader, device)
    record_seed_result(seed, metrics)
    return seed, metrics, history


### 1st Random Seed = 21

In [ ]:
seed, metrics, history = train_or_recover_seed(
    seed=21,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device,
)

if history is not None:
    plot_training_curves(
        epochs=history['epochs'],
        train_loss=history['train_losses'],
        val_loss=history['val_losses'],
        train_acc=history['train_accs'],
        val_acc=history['val_accs'],
        title_prefix=f'Seed {seed}'
    )


### 2nd Random Seed = 42

In [ ]:
seed, metrics, history = train_or_recover_seed(
    seed=42,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device,
)

if history is not None:
    plot_training_curves(
        epochs=history['epochs'],
        train_loss=history['train_losses'],
        val_loss=history['val_losses'],
        train_acc=history['train_accs'],
        val_acc=history['val_accs'],
        title_prefix=f'Seed {seed}'
    )


### 3rd Random Seed = 84

In [ ]:
seed, metrics, history = train_or_recover_seed(
    seed=84,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device,
)

if history is not None:
    plot_training_curves(
        epochs=history['epochs'],
        train_loss=history['train_losses'],
        val_loss=history['val_losses'],
        train_acc=history['train_accs'],
        val_acc=history['val_accs'],
        title_prefix=f'Seed {seed}'
    )


### 4th Random Seed = 168

In [ ]:
seed, metrics, history = train_or_recover_seed(
    seed=168,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device,
)

if history is not None:
    plot_training_curves(
        epochs=history['epochs'],
        train_loss=history['train_losses'],
        val_loss=history['val_losses'],
        train_acc=history['train_accs'],
        val_acc=history['val_accs'],
        title_prefix=f'Seed {seed}'
    )


### 5th Random Seed = 336

In [ ]:
seed, metrics, history = train_or_recover_seed(
    seed=336,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device,
)

if history is not None:
    plot_training_curves(
        epochs=history['epochs'],
        train_loss=history['train_losses'],
        val_loss=history['val_losses'],
        train_acc=history['train_accs'],
        val_acc=history['val_accs'],
        title_prefix=f'Seed {seed}'
    )


## Step 6: Summary of Results

In [ ]:
import csv

seeds = all_results['seeds']
val_accs = [acc*100 for acc in all_results['val_accs']]
val_losses = all_results['val_losses']
control_accs = [acc*100 for acc in all_results['control_accs']]
dementia_accs = [acc*100 for acc in all_results['dementia_accs']]
macro_f1_scores = all_results['macro_f1_scores']
control_f1_scores = all_results['control_f1_scores']
dementia_f1_scores = all_results['dementia_f1_scores']

mean_acc = np.mean(val_accs)
std_acc = np.std(val_accs, ddof=1)

mean_loss = np.mean(val_losses)
std_loss = np.std(val_losses, ddof=1)

mean_control_acc = np.mean(control_accs)
std_control_acc = np.std(control_accs, ddof=1)

mean_dementia_acc = np.mean(dementia_accs)
std_dementia_acc = np.std(dementia_accs, ddof=1)

mean_macro_f1 = np.mean(macro_f1_scores)
std_macro_f1 = np.std(macro_f1_scores, ddof=1)

print(f"\nMean Validation Accuracy: {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"Mean Validation Loss: {mean_loss:.4f} ± {std_loss:.4f}")
print(f"Mean Control Accuracy: {mean_control_acc:.2f}% ± {std_control_acc:.2f}%")
print(f"Mean Dementia Accuracy: {mean_dementia_acc:.2f}% ± {std_dementia_acc:.2f}%")
print(f"Mean Macro F1 Score: {mean_macro_f1:.4f} ± {std_macro_f1:.4f}")

training_fields = [
    "dataset_name",
    "seed",
    "acc",
    "control_acc",
    "dementia_acc",
    "macro_f1",
    "control_f1",
    "dementia_f1",
]
training_rows = []

print(f"\nDetailed Results:")
for i, seed in enumerate(seeds):
    training_row = {
        "dataset_name": DATASET_NAME,
        "seed": seed,
        "acc": f"{all_results['val_accs'][i]:.4f}",
        "control_acc": f"{all_results['control_accs'][i]:.4f}",
        "dementia_acc": f"{all_results['dementia_accs'][i]:.4f}",
        "macro_f1": f"{macro_f1_scores[i]:.4f}",
        "control_f1": f"{control_f1_scores[i]:.4f}",
        "dementia_f1": f"{dementia_f1_scores[i]:.4f}",
    }
    training_rows.append(training_row)
    print(
        f"Seed {seed}: Acc={val_accs[i]:.2f}%, Loss={val_losses[i]:.4f}, "
        f"Control Acc={control_accs[i]:.2f}%, Dementia Acc={dementia_accs[i]:.2f}%, "
        f"Macro F1={macro_f1_scores[i]:.4f}, Control F1={control_f1_scores[i]:.4f}, "
        f"Dementia F1={dementia_f1_scores[i]:.4f}"
    )

TRAINING_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with TRAINING_RESULTS_PATH.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=training_fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(training_rows)

print(f"Saved training results to {TRAINING_RESULTS_PATH}")
training_rows


## Step 7: Test Best Model on Multiple Datasets

In [ ]:
max_acc = max(all_results['val_accs'])
max_acc_indices = [i for i, acc in enumerate(all_results['val_accs']) if acc == max_acc]

if len(max_acc_indices) > 1:
    print(f"Multiple models with accuracy {max_acc*100:.2f}%, selecting one with lowest loss")
    best_idx = min(max_acc_indices, key=lambda i: all_results['val_losses'][i])
else:
    best_idx = max_acc_indices[0]

BEST_SEED = all_results['seeds'][best_idx]
BEST_VAL_ACC = all_results['val_accs'][best_idx] * 100
BEST_VAL_LOSS = all_results['val_losses'][best_idx]
BEST_F1 = all_results['macro_f1_scores'][best_idx]
BEST_MODEL_PATH = MODEL_OUTPUT_DIR / f"seed_{BEST_SEED}" / "best.pth"

In [ ]:
from SLS_Model.model import AD_SLS_Model

test_model = AD_SLS_Model()

checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
test_model.load_state_dict(checkpoint)
test_model = test_model.to(device)
test_model.eval()

REFERENCE_VAL_CSV = PROJECT_ROOT / f"data/processed/{DATASET_NAME}-sls-val.csv"
print(f"Reference VAL_CSV: {REFERENCE_VAL_CSV}")


In [ ]:
LU_TEST_DATASETS = [
    ("Lu", PROJECT_ROOT / "data/raw/Lu"),
    ("Lu-Denoiser", PROJECT_ROOT / "data/denoised/Lu-Denoiser"),
    ("Lu-FRCRN_SE", PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"),
    ("Lu-MossFormer", PROJECT_ROOT / "data/denoised/Lu-MossFormer"),
    ("Lu-Resemble", PROJECT_ROOT / "data/denoised/Lu-Resemble"),
    ("Lu-MAP-SEMamba", PROJECT_ROOT / "data/denoised/Lu-MAP-SEMamba"),
]

OUTPUT_FIELDS = ["dataset", "acc", "control_acc", "dementia_acc", "macro_f1", "control_f1", "dementia_f1"]
METRIC_KEYS = OUTPUT_FIELDS[1:]


def dataset_result_dir(dataset_name: str) -> Path:
    return LU_RESULTS_DIR / dataset_name


def bootstrap_path(dataset_name: str) -> Path:
    output_dir = dataset_result_dir(dataset_name)
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir / f"{dataset_name}_bootstrap_samples.csv"


def complete_rows(path: Path, expected_count: int, required_fields: set[str]):
    if not path.exists():
        return None
    rows = load_dict_rows(path)
    if len(rows) != expected_count:
        print(f"Incomplete existing result: {path} has {len(rows)} rows, expected {expected_count}")
        return None
    for row in rows:
        if not required_fields.issubset(row.keys()):
            print(f"Incomplete existing result: {path} is missing required columns")
            return None
    return rows


def metric_value(row: dict[str, object], key: str) -> float:
    if key == "acc":
        return float(row["accuracy"])
    if key == "macro_f1":
        return (float(row["control_f1"]) + float(row["dementia_f1"])) / 2
    return float(row[key])


def format_metric_row(dataset_name: str, row: dict[str, object]) -> dict[str, str]:
    return {
        "dataset": dataset_name,
        "acc": f"{metric_value(row, 'acc'):.4f}",
        "control_acc": f"{metric_value(row, 'control_acc'):.4f}",
        "dementia_acc": f"{metric_value(row, 'dementia_acc'):.4f}",
        "macro_f1": f"{metric_value(row, 'macro_f1'):.4f}",
        "control_f1": f"{metric_value(row, 'control_f1'):.4f}",
        "dementia_f1": f"{metric_value(row, 'dementia_f1'):.4f}",
    }


def save_metric_rows(path: Path, rows: list[dict[str, object]]) -> None:
    import csv

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=OUTPUT_FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def bootstrap_summary_from_rows(rows: list[dict[str, str]]) -> dict[str, dict[str, float]]:
    summary = {}
    for key in METRIC_KEYS:
        values = np.asarray([float(row[key]) for row in rows], dtype=float)
        values = values[~np.isnan(values)]
        summary[key] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values, ddof=1)),
            "ci95_low": float(np.percentile(values, 2.5)),
            "ci95_high": float(np.percentile(values, 97.5)),
        }
    return summary



LU_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
test_results = []
bootstrap_results = []

for dataset_name, audio_dir in LU_TEST_DATASETS:
    bootstrap_csv = bootstrap_path(dataset_name)
    lu_result = evaluate_lu_with_predictions(
        model=test_model,
        device=device,
        ssl_model=ssl_model,
        batch_size=SLS_BATCH_SIZE,
        dataset_name=dataset_name,
        audio_dir=audio_dir,
        feature_dir_name=f"{dataset_name}_sls_features",
    )
    if lu_result["n_samples"] != LU_SAMPLE_SIZE:
        print(
            f"Warning: {dataset_name} prediction count is {lu_result['n_samples']}; "
            f"bootstrap sample size is configured as {LU_SAMPLE_SIZE}."
        )

    _, raw_bootstrap_sample_rows = bootstrap_binary_metrics_with_samples(
        y_true=lu_result["y_true"],
        y_pred=lu_result["y_pred"],
        n_bootstrap=BOOTSTRAP_REPEATS,
        sample_size=LU_SAMPLE_SIZE,
        seed=BEST_SEED,
    )
    bootstrap_sample_rows = [
        format_metric_row(dataset_name, row)
        for row in raw_bootstrap_sample_rows
    ]
    save_metric_rows(bootstrap_csv, bootstrap_sample_rows)
    bootstrap_summary = bootstrap_summary_from_rows(bootstrap_sample_rows)

    test_row = format_metric_row(dataset_name, lu_result)
    test_results.append(test_row)

    bootstrap_row = {"dataset": dataset_name}
    for key in METRIC_KEYS:
        mean = bootstrap_summary[key]["mean"]
        std = bootstrap_summary[key]["std"]
        bootstrap_row[key] = f"{mean:.4f}±{std:.4f}"
    bootstrap_results.append(bootstrap_row)

    print(f"{dataset_name} result: {test_row}")
    print(f"{dataset_name} bootstrap summary: {bootstrap_summary}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

save_metric_rows(LU_RESULTS_DIR / "test_results.csv", test_results)
save_metric_rows(LU_RESULTS_DIR / "bootstrap_results.csv", bootstrap_results)

print("\n" + "=" * 80)
print("SUMMARY OF ALL TEST RESULTS")
print("=" * 80)
print(f"{'Dataset':<20} {'Acc':<12} {'Macro F1':<12} {'Control Acc':<12} {'Dementia Acc':<12}")
print("-" * 80)
for row in test_results:
    print(f"{row['dataset']:<20} {float(row['acc'])*100:>10.2f}%  {float(row['macro_f1']):>10.4f}  "
          f"{float(row['control_acc'])*100:>10.2f}%  {float(row['dementia_acc'])*100:>10.2f}%")
print("=" * 80)


In [ ]:
test_results_path = LU_RESULTS_DIR / "test_results.csv"
if not test_results_path.exists():
    raise FileNotFoundError(f"Missing test results: {test_results_path}")

test_results = load_dict_rows(test_results_path)

print("\n" + "=" * 90)
print("TEST RESULTS")
print("=" * 90)
for row in test_results:
    print(f"\n[{row['dataset']}]")
    for key in OUTPUT_FIELDS[1:]:
        print(f"{key}: {float(row[key]):.4f}")

test_results
